<a href="https://colab.research.google.com/github/federicasanti/Automated-skin-cancer-detection/blob/main/Final_version_of_the_code_with_best_model_EfficientNetB0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import uuid
import tensorflow as tf

from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.preprocessing.image import save_img
from tensorflow.data import Dataset, AUTOTUNE
from tensorflow.keras.models import Sequential, save_model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Rescaling, Dropout
from tensorflow import Tensor, concat
from tensorflow.keras.callbacks import History, EarlyStopping
from tensorflow.keras.metrics import Precision, Recall, Accuracy
from tensorflow.keras.preprocessing import image_dataset_from_directory
from sklearn.metrics import accuracy_score

RANDOM_SEED = 42

In [ ]:
os.makedirs("models", exist_ok=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data_dir = "/content/drive/MyDrive/Skin_Cancer_Detection/Melanoma_Cancer_Image_Dataset/"
train_ds, val_ds = image_dataset_from_directory(
  data_dir,
  labels='inferred',
  validation_split=0.2,
  image_size=(128, 128),
  subset="both",
  seed=RANDOM_SEED,
  pad_to_aspect_ratio=True,
  shuffle=True,
  batch_size=32)
train_ds

Found 2000 files belonging to 2 classes.
Using 1600 files for training.
Using 400 files for validation.


<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [ ]:
class_names: list[str] = train_ds.class_names
print(class_names)
num_classes = 2

['Benign', 'Malignant']


In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

In [ ]:
es = EarlyStopping(
    monitor="val_loss",
    patience=5,
    verbose=1,
    restore_best_weights=True
)
es

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model

base = EfficientNetB0(weights="imagenet", include_top=False, input_shape=(128,128,3))
base.trainable = False

x = GlobalAveragePooling2D()(base.output)
x = Dense(256, activation="relu")(x)
x = Dropout(0.3)(x)
output = Dense(2, activation="softmax")(x)

effnet_model = Model(inputs=base.input, outputs=output)
effnet_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])


In [ ]:
effnet_model.summary()

history_effnet = effnet_model.fit(train_ds, validation_data=val_ds, epochs=50, callbacks=[es])

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_4         │ (None, 128, 128,  │          0 │ input_layer_2[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_2     │ (None, 128, 128,  │          7 │ rescaling_4[0][0] │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_5         │ (None, 128, 128,  │          0 │ normalization_2[… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 129, 129,  │          0 │ rescaling_5[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 64, 64,    │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 64, 64,    │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 64, 64,    │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 64, 64,    │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 64, 64,    │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 64, 64,    │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 64, 64,    │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 64, 64,    │        512 │ block1a_se_excit

 Total params: 4,378,021 (16.70 MB)

 Trainable params: 328,450 (1.25 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

Epoch 1/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1145s 23s/step - accuracy: 0.7938 - loss: 0.4343 - val_accuracy: 0.9225 - val_loss: 0.2378
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 64s 1s/step - accuracy: 0.9134 - loss: 0.2493 - val_accuracy: 0.9400 - val_loss: 0.2006
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 72s 1s/step - accuracy: 0.9267 - loss: 0.1852 - val_accuracy: 0.9475 - val_loss: 0.1984
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 52s 1s/step - accuracy: 0.9429 - loss: 0.1572 - val_accuracy: 0.9450 - val_loss: 0.2108
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 62s 1s/step - accuracy: 0.9407 - loss: 0.1466 - val_accuracy: 0.9575 - val_loss: 0.1884
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 53s 1s/step - accuracy: 0.9626 - loss: 0.1006 - val_accuracy: 0.9425 - val_loss: 0.1962
Epoch 7/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 80s 1s/step - accuracy: 0.9557 - loss: 0.1090 - val_accuracy: 0.9450 - val_loss: 0.2265
Epoch 8/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 54s 1s/step - accuracy: 0.9587 - loss: 0.1060 - val_accuracy: 0.9450 - val_lo

In [ ]:
malignant_index = class_names.index("Malignant")
print(class_names)
y_pred_prob = effnet_model.predict(val_ds)
y_pred = np.argmax(y_pred_prob, axis=1)
y_malignant_prob = y_pred_prob[:, malignant_index]
threshold = 0.5

y_pred = (y_malignant_prob >= threshold).astype(int)

['Benign', 'Malignant']
13/13 ━━━━━━━━━━━━━━━━━━━━ 17s 1s/step


In [ ]:
from sklearn.metrics import precision_score, recall_score, accuracy_score
y_true = np.concatenate([y.numpy() for _, y in val_ds])

precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='binary')
accuracy = accuracy_score(y_true, y_pred)

print("Precision:", precision)
print("Recall:", recall)
print("Accuracy:", accuracy)
from sklearn.metrics import classification_report

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))

Precision: 0.9575460704264518
Recall: 0.9678899082568807
Accuracy: 0.9575
              precision    recall  f1-score   support

      Benign       0.96      0.95      0.95       182
   Malignant       0.95      0.97      0.96       218

    accuracy                           0.96       400
   macro avg       0.96      0.96      0.96       400
weighted avg       0.96      0.96      0.96       400



In [ ]:
MODEL_PATH = "models/effnet_model.keras"
effnet_model.save(MODEL_PATH)